In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install open_clip_torch huggingface_hub -q

In [ ]:
# =========================================================================
# 🌟 第 0 步：Notebook 专属环境护盾
# =========================================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import open_clip

# 1. 🚀 硬件检测
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📊 硬件检测: {device} (确保显示 cuda！)")

BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 7

# 2. 🌌 BioCLIP 原生图像预处理流水线
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711)
    )
])

# 3. 📂 数据加载
# ⚠️ 关键点：这里一定要填入你之前能成功画出 7 个物种矩阵图的那个极深层路径！
data_dir = '/kaggle/input/datasets/liuziqi77/test1-2/test1.2' 
full_dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =========================================================================
# 🌟 王牌核心魔法：自定义模型包装器 (彻底解决梯度断裂！)
# =========================================================================
class Moss_BioCLIP_FineTuner(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        print("\n🌐 正在把 BioCLIP 2.5 Huge 装入最安全的独立包装器中...")
        base_model, _, _ = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2.5-vith14')
        self.backbone = base_model.visual
        
        # 🔒 绝对冻结骨干网络
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # 🛠️ 独立分类头
        self.custom_head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1024, num_classes)
        )
        
    def forward(self, x):
        # 🛑 显式关闭骨干网络的梯度计算
        with torch.no_grad():
            features = self.backbone(x)
        return self.custom_head(features)

# =========================================================================

# 4. 🧠 实例化并挂载
model = Moss_BioCLIP_FineTuner(num_classes=NUM_CLASSES).to(device)

# 5. ⚖️ 损失函数与优化器
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
# 🎯 优化器只更新我们自己写的 custom_head，绝不抓空！
optimizer = optim.AdamW(model.custom_head.parameters(), lr=2e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 6. 🎯 开启轰鸣
print("\n🚀 包装器密封完毕！开始在云端显卡上安全狂飙...\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.custom_head.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
        
    train_loss = running_loss / total_train
    train_acc = 100.0 * correct_train / total_train
    
    # 盲测
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
            
    val_acc = 100.0 * correct_val / total_val
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    scheduler.step()

# 7. 💾 落地保存
print("\n💾 正在将包装器的最优权重落地保存...")
torch.save(model.state_dict(), "/kaggle/working/best_bioclip_custom.pth")
print("🎉 恭喜！大模型微调完美收官！")

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix
import open_clip

def evaluate_bioclip_on_kaggle():
    # 1. ⚙️ 硬件定位与核心参数（自动对接你右侧存活的权重）
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"📊 启动云端多模态大模型独立评估... 当前设备: {device}")
    
    NUM_CLASSES = 7
    BATCH_SIZE = 16
    # 精准锁定向右侧永久工作区里的 100% 灵魂权重
    WEIGHTS_PATH = "/kaggle/working/best_bioclip_custom.pth" 
    
    # 2. 🌌 构建与训练时完全一致的 BioCLIP 图像预处理流水线（无增强，仅验证缩放）
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.48145466, 0.4578275, 0.40821073),
            std=(0.26862954, 0.26130258, 0.27577711)
        )
    ])
    
    # 3. 📂 加载 Kaggle 数据集（自动读取 7 个物种文件夹并生成正确类名）
    data_dir = '/kaggle/input/datasets/liuziqi77/test1-2/test1.2'
    full_dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
    
    # 严格保持 42 号种子，切出和训练时一模一样的独立测试盲测集
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    _, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
    )
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"✅ 盲测数据加载成功！测试集样本数: {len(val_dataset)} 张")
    
    # 4. 🧠 重新构建和训练时严丝合缝的自定义 BioCLIP 包装器架构
    class Moss_BioCLIP_FineTuner(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
            base_model, _, _ = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2.5-vith14')
            self.backbone = base_model.visual
            self.custom_head = nn.Sequential(
                nn.Dropout(p=0.3),
                nn.Linear(1024, num_classes)
            )
        def forward(self, x):
            with torch.no_grad():
                features = self.backbone(x)
            return self.custom_head(features)
            
    model = Moss_BioCLIP_FineTuner(num_classes=NUM_CLASSES).to(device)
    
    # 5. 🎯 从硬盘中强行注入你的大模型最优微调灵魂
    if os.path.exists(WEIGHTS_PATH):
        model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
        print(f"    ✨ 成功挂载云端最优微调权重: {WEIGHTS_PATH}")
    else:
        print(f"    ❌ 错误：未在工作区找到权重文件 {WEIGHTS_PATH}，请确保右侧 Output 下文件存在！")
        return

    # 6. 开启学术盲测验收模式
    model.eval()
    all_preds = []
    all_labels = []
    
    print("📈 正在对云端盲测集进行高速前向推理...")
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # 7. 📜 生成 SCI 规范的分类学纯净纯文本报告
    display_names = [name.replace('_test', '').replace('1.2', '').strip() for name in full_dataset.classes]
    
    print("\n=================== 📜 BioCLIP 2.5 多模态微调版学术评估战报 ===================")
    report = classification_report(all_labels, all_preds, target_names=display_names, digits=4)
    print(report)
    print("==============================================================================\n")
    
    report_output_path = "/kaggle/working/bioclip_evaluation_report.txt"
    with open(report_output_path, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"💾 纯文本战报已成功导出至: {report_output_path}")
    
    # 8. 🎨 绘制 300 DPI 发表级混淆矩阵（生态绿配色，契合多模态大模型格调）
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(10, 8), dpi=300) # 🌟 开启 300 DPI 高清发表级印刷标准
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Greens) # 大模型换上绿色生态外衣
    plt.title('Confusion Matrix - BioCLIP 2.5 Huge (Linear Probing Framework)', fontsize=13, pad=15)
    plt.colorbar()
    
    tick_marks = np.arange(len(display_names))
    plt.xticks(tick_marks, display_names, rotation=45, ha='right', fontsize=10)
    plt.yticks(tick_marks, display_names, fontsize=10)
    
    # 动态数值着色，防止数字在深绿色背景中融化掉
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontsize=12, weight='bold')
                 
    plt.tight_layout()
    plt.ylabel('True Bryophyte Label (Actual)', fontsize=12, labelpad=10)
    plt.xlabel('Predicted Bryophyte Label (Model)', fontsize=12, labelpad=10)
    
    output_img_path = "/kaggle/working/confusion_matrix_bioclip_v2.5.png"
    plt.savefig(output_img_path, bbox_inches='tight', dpi=300)
    plt.show() # 直接在 Kaggle 屏幕里打出这张高清图！
    
    print(f"🎨 300 DPI 高清矩阵热力图已成功保存至: {output_img_path}")
    print("🎉 恭喜 Peppa！大模型云端独立评估全链路完美闭环！")

if __name__ == "__main__":
    evaluate_bioclip_on_kaggle()